# Query SN Ia or Transient you are interested in 
This is the Jupyter notebook to query transients what you're interested in.<br>

Requirement
- LASAIR: creat account and get API token - https://lasair.lsst.ac.uk/ 
1. Instruction for installation
2. Copy the example notebook on your consol:    git clone https://github.com/lsst-uk/lasair-examples.git  


In [2]:
#In case if you did not install lasair yet
#%pip install lasair

In [2]:
import os, sys, time, json
from lasair import LasairError, lasair_client as lasair
import pandas as pd

API_TOKEN = os.getenv('LASAIR_LSST_TOKEN')
if API_TOKEN is None:
    print("No Token found. Check Spelling. Note that if you have just added your token to your environment variables,     you may need to restart your terminal so your shell settings are reloaded. ")

In [3]:
endpoint = 'https://api.lasair.lsst.ac.uk/api'
L = lasair(API_TOKEN, endpoint=endpoint)

In [4]:
def print15(s):
    print('\n'.join(s.split('\n')[:15]))

In [5]:
selected = 'objects.diaObjectId, objects.ra, objects.decl' 
tables = 'objects,sherlock_classifications'
conditions = 'classification="SN"'
results = L.query(selected, tables, conditions, limit = 8)

for row in results:
    objectId = row['diaObjectId']
    ra = row['ra']
    dec = row['decl']
    print(objectId, ra, dec)

# Just use this objectId in subsequent calls
objectId = results[0]['diaObjectId']

170019695992242795 52.823846701188835 -29.6916002214876
170019696008495213 53.16975409716574 -29.185678374698806
170019696010067995 53.19070564356487 -29.07217739726889
170019696011640884 53.19469873858197 -28.74241923722313
170019696024748213 54.43389733414054 -27.978388168806415
170019696036806674 52.403691729165395 -28.73277626689731
170019696041525298 51.58987643579106 -28.798756629112177
170019696043098131 51.471998267929926 -28.62920326331641


Let's try to match the TNS-classified SN. My interest is SN Ia only. 

In [6]:
selected = """
    objects.diaObjectId,
    crossmatch_tns.tns_name,
    crossmatch_tns.z,
    sherlock_classifications.catalogue_object_id,
    crossmatch_tns.tns_prefix,
    crossmatch_tns.type,
    crossmatch_tns.ra,
    crossmatch_tns.decl,
    objects.absMag,
    objects.nDiaSources
"""

tables = """
    objects,
    watchlist_hits,
    crossmatch_tns,
    sherlock_classifications
"""

conditions = """
    objects.diaObjectId = watchlist_hits.diaObjectId
    AND watchlist_hits.wl_id = 46
    AND watchlist_hits.name = crossmatch_tns.tns_name
    AND crossmatch_tns.type = 'SN Ia'
"""

results = L.query(
    selected,
    tables,
    conditions,
    limit=200
)

In [7]:
df = pd.DataFrame(results)

In [8]:
df

,diaObjectId,tns_name,z,catalogue_object_id,tns_prefix,type,ra,decl,absMag,nDiaSources
0,170019717277810735,2026ctw,0.150000,"10000,342494,11570",SN,SN Ia,151.740293,1.927064,NaN,754
1,170028486191218743,2026qrh,0.100000,"10000,81273,866",SN,SN Ia,64.185342,-48.943229,NaN,88
2,170028500610711681,2024xom,0.044167,CGCG036-024,SN,SN Ia,150.746125,2.336469,NaN,165
3,170028510695916335,2025mbp,0.046160,2MASXJ10062208+0109317,SN,SN Ia,151.592617,1.158385,NaN,273
4,170028511148376326,2021bnv,0.078866,SDSSJ095801.30+034838.6,SN,SN Ia,149.505163,3.809796,99.000,4
...,...,...,...,...,...,...,...,...,...,...
138,313893021499785278,2026ezw,0.131000,SDSSJ095832.59+011610.2,SN,SN Ia,149.635687,1.269344,NaN,20
139,313932566867476549,2026qrh,0.100000,"10000,81273,866",SN,SN Ia,64.185342,-48.943229,NaN,9
140,313963356819030062,2020acmi,0.055000,GALEXASCJ033840.47-271149.6,SN,SN Ia,54.667635,-27.196861,NaN,2
141,313972182793322538,2026fgl,0.164000,WISEAJ095652.35+032811.7,SN,SN Ia,149.218165,3.469919,-16.202,17


In [9]:
from pathlib import Path
workdir = Path("Result")
workdir.mkdir(exist_ok=True)

In [10]:
df.to_parquet(workdir/"snia_results.parquet", index=False)